<img src="logo.png" alt="Vegeta" width="240">

# Svarog-ablation — the guard's ablation as a field problem (FEniCSx → XDMF → PyVista)

The barrier under the stray kilowatt of notebook 49, solved as a finite-element transient in FEniCSx
(`advanced/svarog_ablation_fenicsx.py`: axisymmetric r–z, the apparent-heat-capacity enthalpy method, implicit time
stepping, convection and radiation, ablation by cell deactivation) and rendered from its XDMF with PyVista — against
`hyperion.thermal.heat_plate`, the cheap solver whose verdicts 49 rests on. **Not run here**: dolfinx is a conda-forge /
Docker install ([docs/fenics.md](../docs/fenics.md)); this notebook reads `advanced/runs/<case>/` when a run exists and
says NOT RUN otherwise. The Hyperion reference is computed in place for the comparison.

```bash
micromamba activate fenicsx
python advanced/svarog_ablation_fenicsx.py --material black_pc --thickness 10e-3 --power 933 --radius 17e-3 --t-max 40 --out advanced/runs/pc10
python advanced/svarog_ablation_fenicsx.py --material mild_steel --thickness 1.5e-3 --power 933 --radius 17e-3 --t-max 600 --dt 0.5 --out advanced/runs/steel15
```

In [ ]:
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0, "designs"); sys.path.insert(0, "../advanced")
from vegeta.hyperion import thermal as th
import svarog_ablation_fenicsx as sa
RUNS = Path("../advanced/runs")
CASES = {"pc10": dict(material="black_pc", thickness=10e-3, power=933.0, radius=17e-3, t_max=40.0, dt=0.05, nr=60, nz=20),
         "steel15": dict(material="mild_steel", thickness=1.5e-3, power=933.0, radius=17e-3, t_max=600.0, dt=0.5, nr=60, nz=20)}
try:
    import pyvista as pv
    pv.OFF_SCREEN = True
    HAVE_PV = True
except ImportError:
    HAVE_PV = False
print("dolfinx:", "available" if sa.HAVE_DOLFINX else "not installed (NOT RUN)", "| pyvista:", HAVE_PV)

## 1. Hyperion's reference for each case

The cell model on the same grid resolution and the same material numbers: surface-damage and burn-through times.

In [ ]:
REF = {}
for name, c in CASES.items():
    args = sa.parser().parse_args([f"--{k.replace('_', '-')}={v}" for k, v in c.items()] + ["--reference-only"])
    REF[name] = sa.hyperion_reference(args)
display(pd.DataFrame(REF).T[["solid", "thickness", "surface_damage_time", "failure_time", "failure_mode", "steady", "t_end", "T_max_end"]])

## 2. The FEniCSx runs, when they exist

Each run leaves `result.json` (the times and the centre's log) and `ablation.xdmf` (T, the live indicator, the removal
enthalpy, every tenth step). The log against Hyperion's, then PyVista on the final field.

In [ ]:
FOUND = {}
for name in CASES:
    p = RUNS / name / "result.json"
    if p.exists():
        FOUND[name] = json.loads(p.read_text())
        print(f"{name}: FEniCSx surface damage {FOUND[name]['surface_damage_time']} s, burn-through {FOUND[name]['failure_time']} s "
              f"(Hyperion {REF[name]['surface_damage_time']:.3g} / {REF[name]['failure_time']:.3g} s)")
    else:
        print(f"{name}: NOT RUN — no {p}")
if FOUND:
    fig, axs = plt.subplots(1, len(FOUND), figsize=(7 * len(FOUND), 4.5), squeeze=False)
    for ax, (name, r) in zip(axs[0], FOUND.items()):
        ax.plot(r["log"]["t"], np.array(r["log"]["ablated_depth_centre"]) * 1e3, lw=2, label="FEniCSx: ablated depth at the centre")
        ax.axhline(r["thickness"] * 1e3, color="grey", ls=":", lw=0.8)
        ax.set_xlabel("t [s]"); ax.set_ylabel("depth [mm]"); ax.set_title(name); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
if FOUND and HAVE_PV:
    for name in FOUND:
        xdmf = RUNS / name / "ablation.xdmf"
        try:
            reader = pv.get_reader(str(xdmf)); reader.set_active_time_value(reader.time_values[-1])
            grid = reader.read()[0] if isinstance(reader.read(), pv.MultiBlock) else reader.read()
            pl = pv.Plotter(off_screen=True, window_size=(1100, 500))
            pl.add_mesh(grid.threshold(0.5, scalars="alive"), scalars="T", cmap="inferno", clim=[293, th.solid(CASES[name]["material"].replace("_", " ")).T_fail])
            pl.view_xy(); pl.add_title(f"{name}: temperature of the live material at the last written step")
            pl.screenshot(str(RUNS / name / "final.png")); pl.close()
            from IPython.display import Image; display(Image(str(RUNS / name / "final.png")))
        except Exception as exc:
            print(f"{name}: could not render the XDMF ({type(exc).__name__}: {exc})")
elif FOUND:
    print("PyVista not installed: the XDMF is there, render it with ParaView or install pyvista")

## 3. Reading it

- The cell model and the finite-element model share the physics and the inputs; where they disagree is the grid's
  resolution of the front (Hyperion's 5–10 cells through the plate against the FEM's 20) and the apparent-capacity band.
  A burn-through time within ~15 % validates 49's verdicts; a larger gap means the cheap solver's step or grid, not the
  physics, and the fix is `nz`.
- What the FEM adds that the cell model cannot: a 3-D panel with ribs or a liner, a beam at an angle, a moving beam —
  the enclosure of MVP item M.2 once the panels have a shape.